In [5]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

def parse_semeval_folder(folder_path):
    records = []
    
    for filename in os.listdir(folder_path):
        if not filename.endswith('.xml'):
            continue
            
        filepath = os.path.join(folder_path, filename)
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        question    = root.find('questionText').text.strip()
        reference   = root.find('referenceAnswers/referenceAnswer').text.strip()
        
        for student in root.findall('studentAnswers/studentAnswer'):
            answer = student.text.strip() if student.text else ""
            label  = student.get('accuracy')
            
            if not answer:
                continue
                
            records.append({
                'question'        : question,
                'reference_answer': reference,
                'student_answer'  : answer,
                'label'           : label
            })
    
    return pd.DataFrame(records)

# Paths — update these to match your folder structure
train_path = '../Data/semeval-3way/training/3way/sciEntsBank'
test_path  = '../Data/semeval-3way/test/3way/sciEntsBank/test-unseen-answers'

# Parse
df_train = parse_semeval_folder(train_path)
df_test  = parse_semeval_folder(test_path)

# Check
print(f"Train records : {len(df_train)}")
print(f"Train labels  :\n{df_train['label'].value_counts()}")
print(f"\nTest records  : {len(df_test)}")
print(f"Test labels   :\n{df_test['label'].value_counts()}")

Train records : 4969
Train labels  :
label
incorrect        2462
correct          2008
contradictory     499
Name: count, dtype: int64

Test records  : 540
Test labels   :
label
incorrect        249
correct          233
contradictory     58
Name: count, dtype: int64


In [6]:
df_train.to_csv('../data/semeval_train.csv', index=False)
df_test.to_csv('../data/semeval_test.csv', index=False)
print(f"Train: {len(df_train)} — {df_train['label'].value_counts().to_dict()}")
print(f"Test : {len(df_test)} — {df_test['label'].value_counts().to_dict()}")

Train: 4969 — {'incorrect': 2462, 'correct': 2008, 'contradictory': 499}
Test : 540 — {'incorrect': 249, 'correct': 233, 'contradictory': 58}


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

In [8]:
with open('../outputs/error_profile.json', 'r') as f:
    error_profile = json.load(f)

print(f"Train : {len(df_train)} records")
print(f"Test  : {len(df_test)} records")
print(f"Error profile sections: {list(error_profile.keys())}")

Train : 4969 records
Test  : 540 records
Error profile sections: ['summary', 'substitutions', 'deletions', 'insertions', 'substitution_probs', 'position_analysis', 'word_length_analysis', 'punctuation_analysis']


In [9]:
import random

# Load substitution probabilities
sub_probs   = error_profile['substitution_probs']
del_chars   = error_profile['deletions']
ins_chars   = error_profile['insertions']

# Build deletion and insertion probability lists
del_char_list = list(del_chars.keys())
del_char_weights = list(del_chars.values())

ins_char_list = list(ins_chars.keys())
ins_char_weights = list(ins_chars.values())

# Error type probabilities from IAM
sub_rate = error_profile['summary']['substitution_rate']  # 0.6496
del_rate = error_profile['summary']['deletion_rate']      # 0.0713
ins_rate = error_profile['summary']['insertion_rate']     # 0.2791

def inject_noise(text, corruption_level, seed=42):
    """
    Inject OCR noise into clean text.
    corruption_level: float 0.0 to 1.0 (e.g. 0.1 = 10%)
    """
    random.seed(seed)
    chars  = list(text)
    result = []
    
    num_to_corrupt = int(len(chars) * corruption_level)
    indices_to_corrupt = set(random.sample(
        range(len(chars)), 
        min(num_to_corrupt, len(chars))
    ))
    
    for i, char in enumerate(chars):
        if i not in indices_to_corrupt:
            result.append(char)
            continue
        
        # Decide error type based on IAM rates
        error_type = random.choices(
            ['substitute', 'delete', 'insert'],
            weights=[sub_rate, del_rate, ins_rate]
        )[0]
        
        if error_type == 'substitute':
            char_lower = char.lower()
            if char_lower in sub_probs:
                targets = list(sub_probs[char_lower].keys())
                weights = list(sub_probs[char_lower].values())
                result.append(random.choices(targets, weights=weights)[0])
            else:
                result.append(char)
        
        elif error_type == 'delete':
            pass  # skip character — deletion
        
        elif error_type == 'insert':
            result.append(char)
            ins_char = random.choices(ins_char_list, weights=ins_char_weights)[0]
            result.append(ins_char)
    
    return ''.join(result)

# Test on a sample
sample_text = df_train['student_answer'].iloc[1]
print(f"Original : {sample_text}")
print(f"Noisy 10%: {inject_noise(sample_text, 0.1)}")
print(f"Noisy 20%: {inject_noise(sample_text, 0.2)}")
print(f"Noisy 30%: {inject_noise(sample_text, 0.3)}")

Original : Let the water evaporate and the salt is left behind.
Noisy 10%: Lat thenwater evaborate and the salt is teft be ind.
Noisy 20%: Lht thbsuater baaaorate and the sat is teft behiind.
Noisy 30%: Lyi tttater etraiorate andtthe saht as teft beh iid.


word length corruption weights

In [10]:
def get_word_corruption_rate(word, base_rate):
    """
    Adjust corruption rate based on word length.
    Short words get hit harder — matches IAM word length CER analysis.
    """
    length = len(word)
    
    if length <= 2:
        return min(base_rate * 1.8, 1.0)  # 80% more corruption
    elif length <= 4:
        return min(base_rate * 1.3, 1.0)  # 30% more corruption
    elif length <= 8:
        return base_rate                   # baseline
    else:
        return base_rate * 0.85           # 15% less corruption

# Test
test_words = ['a', 'to', 'the', 'salt', 'water', 'evaporate', 'photosynthesis']
base = 0.20

print("Word length corruption rates at 20% base:")
print()
for word in test_words:
    rate = get_word_corruption_rate(word, base)
    print(f"  '{word}' (len {len(word)}) → {rate:.2f} corruption rate")

Word length corruption rates at 20% base:

  'a' (len 1) → 0.36 corruption rate
  'to' (len 2) → 0.36 corruption rate
  'the' (len 3) → 0.26 corruption rate
  'salt' (len 4) → 0.26 corruption rate
  'water' (len 5) → 0.20 corruption rate
  'evaporate' (len 9) → 0.17 corruption rate
  'photosynthesis' (len 14) → 0.17 corruption rate


Position weights within a word

In [11]:
def get_position_weight(char_idx, word_length):
    """
    Returns corruption probability weight based on 
    character position in word.
    Beginning 36%, Middle 53%, End 11% — from IAM analysis.
    """
    if word_length <= 1:
        return 1.0
    
    relative_pos = char_idx / (word_length - 1)
    
    if relative_pos < 0.25:      # beginning
        return 0.36
    elif relative_pos > 0.75:    # end
        return 0.11
    else:                        # middle
        return 0.53

# Test on word 'evaporate'
word = 'evaporate'
print(f"Position weights for '{word}':")
print()
for i, char in enumerate(word):
    weight = get_position_weight(i, len(word))
    region = 'beginning' if i/(len(word)-1) < 0.25 else 'end' if i/(len(word)-1) > 0.75 else 'middle'
    print(f"  pos {i} '{char}' → weight {weight} ({region})")

Position weights for 'evaporate':

  pos 0 'e' → weight 0.36 (beginning)
  pos 1 'v' → weight 0.36 (beginning)
  pos 2 'a' → weight 0.53 (middle)
  pos 3 'p' → weight 0.53 (middle)
  pos 4 'o' → weight 0.53 (middle)
  pos 5 'r' → weight 0.53 (middle)
  pos 6 'a' → weight 0.53 (middle)
  pos 7 't' → weight 0.11 (end)
  pos 8 'e' → weight 0.11 (end)


Space Deletion Weight

In [12]:
def get_char_corruption_prob(char, position_weight, word_corruption_rate):
    """
    Final corruption probability for a character combining:
    - Word length rate
    - Position weight
    - Space bias (spaces deleted more frequently)
    """
    if char == ' ':
        # Space gets 3x higher deletion probability
        # Space was most deleted character in IAM (3707 times)
        return min(word_corruption_rate * 3.0, 0.8)
    
    # Normal character — combine word rate and position weight
    return word_corruption_rate * position_weight

# Test
test_cases = [
    (' ', 0.53, 0.20),   # space in middle
    ('e', 0.36, 0.20),   # beginning char
    ('a', 0.53, 0.20),   # middle char
    ('e', 0.11, 0.20),   # end char
    ('t', 0.53, 0.36),   # middle char in short word
]

print("Character corruption probabilities:")
print()
for char, pos_weight, word_rate in test_cases:
    prob = get_char_corruption_prob(char, pos_weight, word_rate)
    print(f"  char='{char}' pos_weight={pos_weight} word_rate={word_rate} → prob={prob:.3f}")

Character corruption probabilities:

  char=' ' pos_weight=0.53 word_rate=0.2 → prob=0.600
  char='e' pos_weight=0.36 word_rate=0.2 → prob=0.072
  char='a' pos_weight=0.53 word_rate=0.2 → prob=0.106
  char='e' pos_weight=0.11 word_rate=0.2 → prob=0.022
  char='t' pos_weight=0.53 word_rate=0.36 → prob=0.191


In [13]:
def inject_noise_v2(text, corruption_level, seed=42):
    """
    Final noise injector incorporating:
    1. Word length weighted corruption
    2. Position weighted corruption  
    3. Space deletion weighting
    4. IAM empirical substitution probabilities
    """
    random.seed(seed)
    
    if not isinstance(text, str) or len(text) == 0:
        return text
    
    result = []
    words  = text.split(' ')
    
    for word_idx, word in enumerate(words):
        
        # Add space between words
        if word_idx > 0:
            space_rate = get_char_corruption_prob(' ', 0.53, corruption_level)
            if random.random() > space_rate:
                result.append(' ')
            # else space is deleted — word merging
        
        if len(word) == 0:
            continue
        
        # Get word level corruption rate
        word_rate = get_word_corruption_rate(word, corruption_level)
        
        i = 0
        while i < len(word):
            char = word[i]
            
            # Get position weight
            pos_weight = get_position_weight(i, len(word))
            
            # Get final corruption probability
            corrupt_prob = get_char_corruption_prob(char, pos_weight, word_rate)
            
            if random.random() > corrupt_prob:
                # No corruption
                result.append(char)
                i += 1
                continue
            
            # Decide error type based on IAM rates
            error_type = random.choices(
                ['substitute', 'delete', 'insert'],
                weights=[sub_rate, del_rate, ins_rate]
            )[0]
            
            if error_type == 'substitute':
                char_lower = char.lower()
                if char_lower in sub_probs:
                    targets = list(sub_probs[char_lower].keys())
                    weights = list(sub_probs[char_lower].values())
                    result.append(random.choices(targets, weights=weights)[0])
                else:
                    result.append(char)
            
            elif error_type == 'delete':
                pass  # character dropped
            
            elif error_type == 'insert':
                result.append(char)
                ins_char = random.choices(
                    ins_char_list, 
                    weights=ins_char_weights
                )[0]
                result.append(ins_char)
            
            i += 1
    
    return ''.join(result)

# Test on multiple samples
test_samples = [
    "Let the water evaporate and the salt is left behind.",
    "The harder mineral will scratch the softer one.",
    "Plants use sunlight to produce energy through photosynthesis."
]

print("Noise injection test — v2 (position + length + space weighted)\n")
for text in test_samples:
    print(f"Original : {text}")
    print(f"Noisy 10%: {inject_noise_v2(text, 0.1)}")
    print(f"Noisy 20%: {inject_noise_v2(text, 0.2)}")
    print(f"Noisy 30%: {inject_noise_v2(text, 0.3)}")
    print()

Noise injection test — v2 (position + length + space weighted)

Original : Let the water evaporate and the salt is left behind.
Noisy 10%: Lbt thewafer evhaporate and the saltisleft behind.
Noisy 20%: Lbt tieuaterevaporate and the saltisleftbehind.
Noisy 30%: Lbttieuaterevaporhteandthessttisleftbehind.

Original : The harder mineral will scratch the softer one.
Noisy 10%: Tme harber minerhal will scratch thesofter one.
Noisy 20%: Tme harber minerhal w"ll scratchtoesofterone.
Noisy 30%: Tmehdrdurminherial will scratchthenofteronte.

Original : Plants use sunlight to produce energy through photosynthesis.
Noisy 10%: Ptants inesunhlightto produce endrgythrough photosynthesis.
Noisy 20%: Ptantsinesunhlightto produce endegythrough photosyntoesis.
Noisy 30%: Ptantsinesunhliishttoproduceerergythroughphotosysthesis.



In [14]:
from tqdm import tqdm
import os

tqdm.pandas()

LEVELS = [0.10, 0.20, 0.30]
OUTPUT_DIR = '../data/noisy'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def apply_noise_to_df(df, level, seed=42):
    """Apply inject_noise_v2 to student_answer column at given corruption level."""
    df_noisy = df.copy()
    df_noisy['student_answer_noisy'] = df_noisy['student_answer'].progress_apply(
        lambda text: inject_noise_v2(text, level, seed=seed)
    )
    df_noisy['corruption_level'] = level
    return df_noisy

# Apply to train and test at all 3 levels
for level in LEVELS:
    level_pct = int(level * 100)
    print(f"\n{'='*50}")
    print(f"  Applying {level_pct}% noise...")
    print(f"{'='*50}")

    # Train
    df_train_noisy = apply_noise_to_df(df_train, level)
    train_out = os.path.join(OUTPUT_DIR, f'semeval_train_noisy_{level_pct}.csv')
    df_train_noisy.to_csv(train_out, index=False)
    print(f"  Train saved → {train_out}  ({len(df_train_noisy)} rows)")

    # Test
    df_test_noisy = apply_noise_to_df(df_test, level)
    test_out = os.path.join(OUTPUT_DIR, f'semeval_test_noisy_{level_pct}.csv')
    df_test_noisy.to_csv(test_out, index=False)
    print(f"  Test  saved → {test_out}   ({len(df_test_noisy)} rows)")

print("\nDone! Files saved:")
for level in LEVELS:
    pct = int(level * 100)
    print(f"  noisy_{pct}/  →  semeval_train_noisy_{pct}.csv  |  semeval_test_noisy_{pct}.csv")


  Applying 10% noise...


100%|██████████| 4969/4969 [00:00<00:00, 13922.35it/s]


  Train saved → ../data/noisy\semeval_train_noisy_10.csv  (4969 rows)


100%|██████████| 540/540 [00:00<00:00, 8301.32it/s]


  Test  saved → ../data/noisy\semeval_test_noisy_10.csv   (540 rows)

  Applying 20% noise...


...

  Test  saved → ../data/noisy\semeval_test_noisy_20.csv   (540 rows)

  Applying 30% noise...


100%|██████████| 4969/4969 [00:00<00:00, 10290.49it/s]


  Train saved → ../data/noisy\semeval_train_noisy_30.csv  (4969 rows)


100%|██████████| 540/540 [00:00<00:00, 7182.78it/s]

  Test  saved → ../data/noisy\semeval_test_noisy_30.csv   (540 rows)

Done! Files saved:
  noisy_10/  →  semeval_train_noisy_10.csv  |  semeval_test_noisy_10.csv
  noisy_20/  →  semeval_train_noisy_20.csv  |  semeval_test_noisy_20.csv
  noisy_30/  →  semeval_train_noisy_30.csv  |  semeval_test_noisy_30.csv


In [21]:
import pandas as pd

# Load one noisy file and verify
df_check = pd.read_csv('../data/noisy/semeval_train_noisy_20.csv')

print(f"Columns: {df_check.columns.tolist()}")
print(f"Rows: {len(df_check)}")
print()

# Show 3 examples
for i in range(3):
    print(f"Original : {df_check['student_answer'].iloc[i]}")
    print(f"Noisy 20%: {df_check['student_answer_noisy'].iloc[i]}")
    print(f"Label    : {df_check['label'].iloc[i]}")
    print()

Columns: ['question', 'reference_answer', 'student_answer', 'label', 'student_answer_noisy', 'corruption_level']
Rows: 4969

Original : By letting it sit in a dish for a day.
Noisy 20%: Bt lerting itsihtinaa dishforadao.
Label    : incorrect

Original : Let the water evaporate and the salt is left behind.
Noisy 20%: Lbt tieuaterevaporate and the saltisleftbehind.
Label    : correct

Original : The water evaporated and left salt crystals.
Noisy 20%: Tme wdterevaporhatedand leftsalt trystals.
Label    : correct



Phase 1 :



In [15]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ── Directories ──────────────────────────────────────────────
MODELS_DIR  = '../models'
RESULTS_DIR = '../results/phase1'

os.makedirs(f'{MODELS_DIR}/sbert',   exist_ok=True)
os.makedirs(f'{MODELS_DIR}/bert',    exist_ok=True)
os.makedirs(f'{MODELS_DIR}/roberta', exist_ok=True)
os.makedirs(RESULTS_DIR,             exist_ok=True)

# ── Load all data ─────────────────────────────────────────────
df_train = pd.read_csv('../data/semeval_train.csv')          # clean train
df_test  = pd.read_csv('../data/semeval_test.csv')           # clean test

df_test_n10 = pd.read_csv('../data/noisy/semeval_test_noisy_10.csv')
df_test_n20 = pd.read_csv('../data/noisy/semeval_test_noisy_20.csv')
df_test_n30 = pd.read_csv('../data/noisy/semeval_test_noisy_30.csv')

TEST_SETS = {
    'clean'   : (df_test,    'student_answer'),
    'noisy_10': (df_test_n10,'student_answer_noisy'),
    'noisy_20': (df_test_n20,'student_answer_noisy'),
    'noisy_30': (df_test_n30,'student_answer_noisy'),
}

# ── Label encoder (shared across all models) ──────────────────
le = LabelEncoder()
y_train = le.fit_transform(df_train['label'])

with open(f'{MODELS_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Classes:", le.classes_)
print("Train size:", len(df_train))
print("Test sets loaded:", list(TEST_SETS.keys()))

Classes: ['contradictory' 'correct' 'incorrect']
Train size: 4969
Test sets loaded: ['clean', 'noisy_10', 'noisy_20', 'noisy_30']


SBert

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

print("Loading SBERT...")
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def get_sbert_features(df, student_col='student_answer'):
    """Extract 4 similarity features per row."""
    refs      = df['reference_answer'].tolist()
    students  = df[student_col].tolist()
    questions = df['question'].tolist()

    print("  Encoding reference answers...")
    emb_ref  = sbert.encode(refs,      batch_size=64, show_progress_bar=True)
    print("  Encoding student answers...")
    emb_stu  = sbert.encode(students,  batch_size=64, show_progress_bar=True)
    print("  Encoding questions...")
    emb_ques = sbert.encode(questions, batch_size=64, show_progress_bar=True)

    # Feature 1: cosine sim (ref vs student)  — main signal
    sim_ref_stu  = np.array([cos_sim([r],[s])[0][0] for r,s in zip(emb_ref, emb_stu)])
    # Feature 2: cosine sim (question vs student)
    sim_que_stu  = np.array([cos_sim([q],[s])[0][0] for q,s in zip(emb_ques, emb_stu)])
    # Feature 3: length ratio (how verbose is student vs reference)
    len_ratio    = np.array([len(s.split())/max(len(r.split()),1) for s,r in zip(students, refs)])
    # Feature 4: squared sim (non-linear signal)
    sim_sq       = sim_ref_stu ** 2

    features = np.column_stack([sim_ref_stu, sim_que_stu, len_ratio, sim_sq])
    return features, emb_ref, emb_stu

# ── Extract train features & fit classifier ───────────────────
print("\n[ SBERT ] Extracting train features...")
X_train_sbert, _, _ = get_sbert_features(df_train)

clf_sbert = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf_sbert.fit(X_train_sbert, y_train)
print("Classifier trained.")

# ── Save ──────────────────────────────────────────────────────
with open(f'{MODELS_DIR}/sbert/classifier.pkl', 'wb') as f:
    pickle.dump(clf_sbert, f)

# Save model name so frontend knows which SBERT to load
with open(f'{MODELS_DIR}/sbert/config.json', 'w') as f:
    json.dump({'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
               'features': ['sim_ref_stu', 'sim_que_stu', 'len_ratio', 'sim_sq']}, f)

print("SBERT model saved.")

# ── Evaluate on all test sets ─────────────────────────────────
sbert_results = {}

for split_name, (df_split, student_col) in TEST_SETS.items():
    print(f"\n  Evaluating on {split_name}...")
    X_test, _, _ = get_sbert_features(df_split, student_col)
    y_true = le.transform(df_split['label'])
    y_pred = clf_sbert.predict(X_test)

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='weighted')
    report = classification_report(y_true, y_pred, target_names=le.classes_, output_dict=True)

    sbert_results[split_name] = {'accuracy': acc, 'f1_weighted': f1, 'report': report}
    print(f"  Accuracy: {acc:.4f}  |  F1: {f1:.4f}")

with open(f'{RESULTS_DIR}/sbert_results.json', 'w') as f:
    json.dump(sbert_results, f, indent=2)

print("\nSBERT results saved.")

Loading SBERT...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  Encoding student answers...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

  Encoding questions...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

  Accuracy: 0.4611  |  F1: 0.2910

SBERT results saved.


Bert Fine Tuning

RoBert Fine Tuning

Summary Table + Plot

In [ ]:
import matplotlib.pyplot as plt

# ── Build summary DataFrame ───────────────────────────────────
rows = []
for model_name, results in [('SBERT', sbert_results),
                              ('BERT',  bert_results),
                              ('RoBERTa', roberta_results)]:
    for split, metrics in results.items():
        rows.append({
            'model'      : model_name,
            'test_split' : split,
            'accuracy'   : round(metrics['accuracy'], 4),
            'f1_weighted': round(metrics['f1_weighted'], 4),
        })

df_results = pd.DataFrame(rows)
df_results.to_csv(f'{RESULTS_DIR}/all_results.csv', index=False)
print(df_results.pivot(index='model', columns='test_split', values='accuracy'))

# ── Plot degradation curves ───────────────────────────────────
splits      = ['clean', 'noisy_10', 'noisy_20', 'noisy_30']
x_labels    = ['Clean (0%)', 'Noisy 10%', 'Noisy 20%', 'Noisy 30%']
colors      = {'SBERT': '#4C72B0', 'BERT': '#DD8452', 'RoBERTa': '#55A868'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model_name, results in [('SBERT', sbert_results),
                              ('BERT',  bert_results),
                              ('RoBERTa', roberta_results)]:
    accs = [results[s]['accuracy']   for s in splits]
    f1s  = [results[s]['f1_weighted'] for s in splits]
    ax1.plot(x_labels, accs, marker='o', label=model_name, color=colors[model_name], linewidth=2)
    ax2.plot(x_labels, f1s,  marker='s', label=model_name, color=colors[model_name], linewidth=2)

ax1.set_title('Accuracy vs Noise Level (Phase 1)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title('Weighted F1 vs Noise Level (Phase 1)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Weighted F1')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase1_degradation_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")